# 🎓 PI-LSTM Training — Easy Kaggle Runner

**What this does (3 sentences):**
1. This notebook trains your PI-LSTM physics model on Kaggle's free GPU — all the confusing setup is already done for you.
2. You pick **one letter** (A, B, C, or SMOKE) in the single editable cell, and everything else runs by itself.
3. When it finishes, the notebook tells you exactly which files to download and where to put them on your computer.

⏱️ **Total time:** SMOKE ≈ 15 minutes · A / B / C ≈ 3 hours each on the free GPU · EVIDENCE ≈ 30–60 minutes.

👉 **Just press `Run All`.** The only cell you ever edit is marked with ⭐ (Cell 3). If anything goes wrong, the notebook prints a plain-English ❌ message telling you what to do.


In [ ]:
# Cell 1 — SETUP: unpack the project zip + check everything is here (no edits needed)
import glob, os, sys, zipfile

WORK = '/kaggle/working'
PROJECT = f'{WORK}/IsotopePINN'

try:
    import torch
    if torch.cuda.is_available():
        print('GPU found:', torch.cuda.get_device_name(0))
    else:
        print('⚠️ No GPU yet — the pre-flight cell below will remind you to turn it on.')
except Exception as e:
    print('⚠️ torch import hiccup (pre-flight will re-check):', e)

print('\n=== Looking for your uploaded project in /kaggle/input ===')
os.makedirs(PROJECT, exist_ok=True)

def _safe(name):
    name = name.replace('\\', '/').strip()
    if not name or name.startswith('/') or '..' in name.split('/'):
        return None
    return name

candidates = glob.glob('/kaggle/input/**/IsotopePINN_Project.zip', recursive=True)
if not candidates:
    candidates = [
        p for p in glob.glob('/kaggle/input/**/*.zip', recursive=True)
        if 'isotope' in p.lower() or 'pinn' in p.lower()
    ]

if candidates:
    ZIP = candidates[0]
    print('Using zip:', ZIP)
    extracted = 0
    with zipfile.ZipFile(ZIP) as zf:
        bad = zf.testzip()
        if bad is not None:
            raise ValueError(f'❌ The zip is damaged (bad file: {bad}). Rebuild it on your PC with build_colab_zip.py and re-upload.')
        for m in zf.infolist():
            if m.is_dir():
                continue
            rel = _safe(m.filename)
            if not rel:
                continue
            if rel.endswith('v3_pilstm/weights/pi_lstm_best.pth'):
                print('Skipping old weights file (you will train fresh ones):', rel)
                continue
            dest = os.path.join(PROJECT, rel)
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            with zf.open(m) as src, open(dest, 'wb') as dst:
                dst.write(src.read())
            extracted += 1
    print('Unpacked', extracted, 'files')
else:
    # Kaggle sometimes auto-unzips dataset zips — copy the tree into /kaggle/working
    roots = []
    for marker in glob.glob('/kaggle/input/**/v3_pilstm/train_pi_lstm.py', recursive=True):
        roots.append(os.path.dirname(os.path.dirname(marker)))
    if not roots:
        raise FileNotFoundError(
            '❌ No project found. Did you attach your dataset? '
            'Right panel -> Add Data -> pick your isotope-pinn dataset.'
        )
    SRC = roots[0]
    print('Using already-unpacked dataset at:', SRC)
    import shutil
    copied = 0
    for dirpath, dirnames, filenames in os.walk(SRC):
        rel_dir = os.path.relpath(dirpath, SRC)
        for fn in filenames:
            rel = fn if rel_dir == '.' else os.path.join(rel_dir, fn).replace('\\', '/')
            if rel.endswith('v3_pilstm/weights/pi_lstm_best.pth'):
                continue
            dest = os.path.join(PROJECT, rel)
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            shutil.copy2(os.path.join(dirpath, fn), dest)
            copied += 1
    print('Copied', copied, 'files')

print('\n=== Checking the project is complete and NEW enough ===')
required = [
    'pinn_model.py', 'ra226_ac225_transmutation.py', 'weights/pinn_best_weights.pth',
    'v3_pilstm/train_pi_lstm.py', 'v3_pilstm/models/pi_lstm.py',
    'v3_pilstm/data/trajectory_dataset.py', 'v3_pilstm/physics/distill.py',
    'v3_pilstm/analysis/endpoint_eval.py', 'v3_pilstm/analysis/train_baseline.py',
    'v3_pilstm/analysis/compare_models.py',
    'v3_pilstm/analysis/run_conformal_validation.py',
    'v3_pilstm/scripts/speed_benchmark.py',
]
missing = [f for f in required if not os.path.exists(os.path.join(PROJECT, f))]
if missing:
    raise FileNotFoundError(
        '❌ Zip incomplete — rebuild with build_colab_zip.py on your PC and re-upload. Missing: ' + str(missing)
    )

# The v2 physics data folder only exists in freshly-built zips.
if not os.path.isdir(os.path.join(PROJECT, 'data/evaluated')):
    raise FileNotFoundError(
        '❌ The folder data/evaluated/ is missing — you uploaded the OLD zip.\n'
        '   Fix: on your PC run  python build_colab_zip.py  then upload the NEW '
        'IsotopePINN_Project.zip as your Kaggle dataset.'
    )
print('data/evaluated/ found ✔ (new zip confirmed)')

train_py = open(f'{PROJECT}/v3_pilstm/train_pi_lstm.py', encoding='utf-8').read()
if 'val Ac225 med' not in train_py:
    raise RuntimeError('❌ OLD train_pi_lstm.py — rebuild the zip on your PC with build_colab_zip.py and re-upload.')
if 'PI_LSTM_LOSS' not in train_py:
    raise RuntimeError('❌ train_pi_lstm.py is too old (no expmix loss) — rebuild the zip on your PC and re-upload.')
print('train_pi_lstm.py is up to date ✔')

os.chdir(PROJECT)
for p in (PROJECT, f'{PROJECT}/v3_pilstm'):
    if p in sys.path:
        sys.path.remove(p)
    sys.path.insert(0, p)

print('\n✅ SETUP DONE — project ready at', PROJECT)
print('Now go to the ⭐ cell below and pick your letter.')


In [ ]:
# ⭐⭐⭐ THE ONLY CELL YOU EDIT ⭐⭐⭐
#
# Change the ONE letter below, then press Run All. That's it.
#
#   RUN = "SMOKE"    🔬 Test run — checks everything works (NOT real results) ..... ~15 minutes
#   RUN = "A"        🆕 New model: new physics data + expmix loss + new scenario .. ~3 hours
#   RUN = "B"        🎛️  Control run: original settings (old data + trap loss) ..... ~3 hours
#   RUN = "C"        📏 Baseline LSTM (no physics) — the fair-comparison model .... ~3 hours
#   RUN = "EVIDENCE" 📊 No training — builds your evidence numbers (run AFTER A, B, C) ~30–60 min
#
# Recommended order for your science project:  SMOKE → A → B → C → EVIDENCE

RUN = "A"   # 👈 change this letter only

# ------------------ everything below is automatic — do not edit ------------------

_PROJECT = globals().get('PROJECT', '/kaggle/working/IsotopePINN')

# Shared "quality" training recipe (the settings that produced the best results).
_QUALITY = {
    'PILSTM_CKPT_METRIC': 'endpoint_ac225',
    'PILSTM_EPOCHS': '6000',
    'PILSTM_PRETRAIN_FRAC': '0.20',
    'PILSTM_GRAD_BALANCE': '1',
    'PILSTM_DISTILL': '1',
    'PILSTM_DISTILL_WEIGHT': '5',
    'PILSTM_DATA_WEIGHT': '35',
    'PILSTM_PHYSICS_WEIGHT': '20',
    'PILSTM_MASS_WEIGHT': '10',
    'PILSTM_CAUSAL_EPS': '2.5',
    'PILSTM_N_TRAIN': '1400',
    'PILSTM_N_STEPS': '64',
    'PILSTM_N_VAL': '22',
    'PILSTM_N_TEST': '22',
    'PILSTM_BATCH': '16',
    'PILSTM_HIDDEN': '256',
    'PILSTM_FOURIER': '8',
    'PILSTM_TIME_FOURIER': '16',
    'PILSTM_HARD_IC': '1',
    'PILSTM_OVERSHOOT_WEIGHT': '20',
    'PILSTM_LBFGS_ITER': '60',
    'PILSTM_LOG_WEIGHT': '2.0',
    'PILSTM_LOG_EVERY': '10',
    'PILSTM_FLOAT64': '1',      # high-precision math (matches the best run)
    'PILSTM_EVAL_EVERY': '1',   # check progress every epoch (matches the best run)
    'PILSTM_EARLY_STOP': '0',
    'PI_LSTM_SEED': '42',
    'PI_LSTM_ADAPTIVE_WEIGHTS': '0',
    'PI_LSTM_CURRICULUM': '0',
}

_W = f'{_PROJECT}/v3_pilstm/weights'
_R = f'{_PROJECT}/v3_pilstm/results'

def _redirected(tag):
    '''Keep each run's output files separate so nothing gets overwritten.'''
    return {
        'PILSTM_WEIGHTS_PATH': f'{_W}/pi_lstm_best_{tag}.pth',
        'PILSTM_STATE_PATH': f'{_W}/pi_lstm_train_state_{tag}.pth',
        'PILSTM_PROGRESS_PATH': f'{_R}/train_progress_{tag}.json',
        'PILSTM_RESULTS_PATH': f'{_R}/train_summary_{tag}.json',
    }

_RUNS = {
    'A': {
        'script': 'v3_pilstm/train_pi_lstm.py',
        'env': {**_QUALITY,
                'ODE_DATA_VERSION': 'v2',      # new evaluated nuclear data
                'PI_LSTM_LOSS': 'expmix',      # new exp-mixture physics loss
                'SCENARIO_VERSION': 'v2',      # true 1-gram inventory
                'PI_LSTM_SEED': '42'},
        # Run A is the champion: it writes the standard file names that the
        # evidence scripts look for (pi_lstm_best.pth / train_summary.json).
        'weights': 'v3_pilstm/weights/pi_lstm_best.pth',
        'summary': 'v3_pilstm/results/train_summary.json',
        'desc': '🆕 NEW MODEL — new physics data (v2), expmix loss, new scenario (v2), seed 42',
        'time': '~3 hours',
    },
    'B': {
        'script': 'v3_pilstm/train_pi_lstm.py',
        'env': {**_QUALITY,
                'ODE_DATA_VERSION': 'v1',      # legacy physics data
                'PI_LSTM_LOSS': 'trap',        # legacy trapezoid loss
                'SCENARIO_VERSION': 'v1',      # legacy inventory
                'PI_LSTM_SEED': '42',
                **_redirected('B')},
        'weights': 'v3_pilstm/weights/pi_lstm_best_B.pth',
        'summary': 'v3_pilstm/results/train_summary_B.json',
        'desc': '🎛️ CONTROL — original settings (v1 data, trap loss, v1 scenario), seed 42',
        'time': '~3 hours',
    },
    'C': {
        'script': 'v3_pilstm/analysis/train_baseline.py',
        'env': {'BASELINE_EPOCHS': '6000',
                'SCENARIO_VERSION': 'v2',      # same data as Run A for a fair fight
                'ODE_DATA_VERSION': 'v2',
                'PI_LSTM_SEED': '42'},
        'weights': 'v3_pilstm/weights/baseline_lstm_best.pth',
        'summary': 'v3_pilstm/results/baseline_lstm_summary.json',
        'desc': '📏 BASELINE LSTM — same data, but NO physics (the fair-comparison model)',
        'time': '~3 hours',
    },
    'SMOKE': {
        'script': 'v3_pilstm/train_pi_lstm.py',
        'env': {**_QUALITY,
                'ODE_DATA_VERSION': 'v2',
                'PI_LSTM_LOSS': 'expmix',
                'SCENARIO_VERSION': 'v2',
                'PI_LSTM_SEED': '42',
                'PILSTM_EPOCHS': '150',        # tiny — just to prove the setup works
                'PILSTM_FLOAT64': '0',         # faster math for the test run
                'PILSTM_EVAL_EVERY': '25',
                'PILSTM_BATCH': '32',
                'PILSTM_GRAD_BALANCE': '0',
                'PILSTM_LBFGS_ITER': '0',
                **_redirected('SMOKE')},
        'weights': 'v3_pilstm/weights/pi_lstm_best_SMOKE.pth',
        'summary': 'v3_pilstm/results/train_summary_SMOKE.json',
        'desc': '🔬 SMOKE TEST — same as A but only 150 epochs (results are NOT for your poster)',
        'time': '~15 minutes',
    },
    'EVIDENCE': {
        'script': None,  # no training — the evidence cell does the work
        'env': {'ODE_DATA_VERSION': 'v2',
                'SCENARIO_VERSION': 'v2',
                'PILSTM_FLOAT64': '1',
                'CONFORMAL_MODE': 'cv+',
                'CONFORMAL_N_CAL': '100',
                'CONFORMAL_N_TEST': '100',
                'CONFORMAL_MODEL': 'pilstm'},
        'weights': None,
        'summary': None,
        'desc': '📊 EVIDENCE — compares your models + uncertainty + speed (no training)',
        'time': '~30–60 minutes',
    },
}

if RUN not in _RUNS:
    raise ValueError(
        f'❌ RUN = {RUN!r} is not a choice. Use exactly one of: '
        '"A", "B", "C", "SMOKE", "EVIDENCE"  (capital letters, with quotes).'
    )

_CFG = _RUNS[RUN]
RUN_ENV = _CFG['env']
TRAIN_SCRIPT = _CFG['script']
RUN_WEIGHTS = _CFG['weights']
RUN_SUMMARY = _CFG['summary']

print('You picked RUN =', RUN)
print(_CFG['desc'])
print('Expected time:', _CFG['time'])
if RUN == 'SMOKE':
    print('Note: SMOKE is only a rehearsal — its numbers are NOT for your poster.')
if RUN == 'EVIDENCE':
    print('Note: EVIDENCE does not train — it needs A, B and C finished first (in this session, or with their files copied back in).')
print('\nSettings that will be used:')
for k in sorted(RUN_ENV):
    print(f'  {k} = {RUN_ENV[k]}')
print('\n✅ Choice locked in — run the next cell.')


In [ ]:
# Cell 3 — PRE-FLIGHT CHECK (no edits needed): makes sure everything is ready BEFORE the long run
import os, sys

if 'RUN' not in globals() or 'RUN_ENV' not in globals():
    raise RuntimeError('❌ You skipped the ⭐ cell above. Go back and run it first.')

_PROJECT = globals().get('PROJECT', '/kaggle/working/IsotopePINN')
_problems = []

# 1) Settings make sense
if RUN != 'EVIDENCE':
    if RUN_ENV.get('ODE_DATA_VERSION') not in ('v1', 'v2'):
        _problems.append('ODE_DATA_VERSION must be v1 or v2 — re-run the ⭐ cell (do not edit its lower half).')
    if RUN != 'C' and RUN_ENV.get('PI_LSTM_LOSS') not in ('trap', 'expmix'):
        _problems.append('PI_LSTM_LOSS must be trap or expmix — re-run the ⭐ cell.')
    if RUN_ENV.get('SCENARIO_VERSION') not in ('v1', 'v2'):
        _problems.append('SCENARIO_VERSION must be v1 or v2 — re-run the ⭐ cell.')
    _ep = RUN_ENV.get('PILSTM_EPOCHS', RUN_ENV.get('BASELINE_EPOCHS', ''))
    if not str(_ep).isdigit() or int(_ep) <= 0:
        _problems.append('Epoch count is not a positive number — re-run the ⭐ cell.')
print('1) Settings sanity ...', '✅' if not _problems else '❌')

# 2) GPU is on
try:
    import torch
    _gpu_ok = torch.cuda.is_available()
except Exception:
    _gpu_ok = False
if _gpu_ok:
    print('2) GPU ... ✅', torch.cuda.get_device_name(0))
else:
    print('2) GPU ... ❌ turn on GPU in Settings (right panel → Settings → Accelerator → GPU), then Restart & Run All')
    _problems.append('GPU is off.')

# 3) Data actually loads (build a tiny practice batch — takes ~30 seconds)
if RUN != 'EVIDENCE' and _gpu_ok:
    for _k in ('ODE_DATA_VERSION', 'SCENARIO_VERSION', 'PI_LSTM_SEED'):
        if _k in RUN_ENV:
            os.environ[_k] = RUN_ENV[_k]
    try:
        from v3_pilstm.data.trajectory_dataset import build_dataloaders
        _tl, _vl, _sl, _ = build_dataloaders(
            n_train=6, n_val=2, n_test=2, n_steps=16, batch_size=4,
            seed=int(RUN_ENV.get('PI_LSTM_SEED', '42')),
            scenario_version=RUN_ENV.get('SCENARIO_VERSION', 'v1'),
        )
        _batch = next(iter(_tl))
        _key = sorted(_batch.keys())[0]
        print('3) Data loading ... ✅ practice batch loaded,', _key, 'shape', tuple(_batch[_key].shape))
    except Exception as _e:
        print('3) Data loading ... ❌', type(_e).__name__, _e)
        _problems.append('Data failed to load. If you changed anything in the ⭐ cell below the menu, undo it. Otherwise rebuild the zip with build_colab_zip.py and re-upload.')
elif RUN == 'EVIDENCE':
    _need = ['v3_pilstm/weights/pi_lstm_best.pth',
             'v3_pilstm/weights/baseline_lstm_best.pth',
             'weights/pinn_best_weights.pth']
    _miss = [f for f in _need if not os.path.isfile(os.path.join(_PROJECT, f))]
    if _miss:
        print('3) Evidence inputs ... ❌ missing:', _miss)
        print('   Fix: finish runs A and C first (in this same session), or copy their weight files into the project.')
        _problems.append('Evidence inputs missing.')
    else:
        print('3) Evidence inputs ... ✅ all model files found')

if _problems:
    print('\n❌ NOT READY:')
    for _p in _problems:
        print('  -', _p)
    raise SystemExit('Fix the ❌ items above, then re-run this cell.')

print('\n✅ READY — now run the next cell.')


In [ ]:
# Cell 4 — TRAINING (no edits needed): runs your chosen run and babysits it for you
import os, re, subprocess, sys

if 'RUN' not in globals() or 'RUN_ENV' not in globals():
    raise RuntimeError('❌ You skipped the ⭐ cell. Go back and run it first.')

_PROJECT = globals().get('PROJECT', '/kaggle/working/IsotopePINN')
_WORK = globals().get('WORK', '/kaggle/working')

if RUN == 'EVIDENCE':
    print('RUN = "EVIDENCE" — nothing to train. ✅ Skip ahead to the last cell.')
else:
    # The babysitter understands log lines like:
    #   epoch   600/6000 | loss=1.23e+03 data=1.10e+03 phys=1.30e+02 | val Ac225 med=0.0512 best=0.0499@580
    _LOG_RE = re.compile(
        r'epoch\s+(\d+)/(\d+)\s*\|\s*'
        r'loss=([0-9.eE+-]+|nan|inf|-inf)\s+'
        r'data=([0-9.eE+-]+|nan|inf|-inf)\s+'
        r'phys=([0-9.eE+-]+|nan|inf|-inf)\s*\|\s*'
        r'val Ac225 med=([0-9.]+|n/a)\s+best=([0-9.]+|inf|nan)@(\d+)'
    )
    _BASE_RE = re.compile(
        r'epoch\s+(\d+)/(\d+)\s*\|\s*data_loss=([0-9.eE+-]+|nan|inf|-inf)'
    )

    _losses = []          # rolling window of recent loss values
    _warned_nan = 0
    _warned_rise_at = -10**9
    _warned_phys = False
    _warned_stuck_at = -10**9

    def _f(x):
        try:
            return float(x)
        except ValueError:
            return float('nan')

    def _babysit(line):
        global _warned_nan, _warned_rise_at, _warned_phys, _warned_stuck_at
        m = _LOG_RE.search(line)
        if m:
            ep, total = int(m.group(1)), int(m.group(2))
            loss, data, phys = _f(m.group(3)), _f(m.group(4)), _f(m.group(5))
            val = _f(m.group(6))
            best, best_ep = _f(m.group(7)), int(m.group(8))

            if loss != loss or loss in (float('inf'), float('-inf')) or data != data or phys != phys:
                _warned_nan += 1
                if _warned_nan <= 3:
                    print('\n🛑 RUN IS BROKEN — stop now (press the Stop button), tell the assistant: NaN loss\n')
                return
            _losses.append(loss)
            if len(_losses) > 10:
                _losses.pop(0)
            if len(_losses) == 10 and _losses[-1] > _losses[0] * 1.05 and ep - _warned_rise_at >= 200:
                _warned_rise_at = ep
                print('\n⚠️ loss going up over the last 10 logs — likely divergence; consider stopping.\n')
                return
            if ep < 500 and data > 0 and phys > 50 * data and not _warned_phys:
                _warned_phys = True
                print('\n⚠️ physics loss dominating (phys is 50x data) — retune PILSTM_PHYSICS_WEIGHT (ask assistant).\n')
                return
            if (ep >= 2000 and (ep - best_ep) >= 1500 and val == val and val > 0.15
                    and ep - _warned_stuck_at >= 500):
                _warned_stuck_at = ep
                print('\n⚠️ looks stuck — best score has not improved in 1500+ epochs and val Ac225 med is still '
                      f'{val:.4f}. Compare with a previous run before spending more time.\n')
                return
            if ep % 500 == 0 or ep == total:
                _v = f'{val:.4f}' if val == val else 'n/a'
                print(f'💚 healthy — val Ac225 med={_v}, best={best:.4f}@{best_ep} (epoch {ep}/{total})')
            return
        b = _BASE_RE.search(line)
        if b:
            ep, total, dl = int(b.group(1)), int(b.group(2)), _f(b.group(3))
            if dl != dl or dl in (float('inf'), float('-inf')):
                print('\n🛑 RUN IS BROKEN — stop now (press the Stop button), tell the assistant: NaN loss\n')
            elif ep % 500 == 0 or ep == total:
                print(f'💚 healthy — baseline epoch {ep}/{total}, data_loss={dl:.4e}')

    _env = os.environ.copy()
    _env.update(RUN_ENV)
    _env['PYTHONUNBUFFERED'] = '1'
    _log_path = f'{_WORK}/train_log_RUN_{RUN}.txt'
    print('Starting:', TRAIN_SCRIPT)
    print('Full log also saved to:', _log_path)
    print('Go have a snack — this takes about', _RUNS[RUN]['time'], '\n')

    with open(_log_path, 'w', encoding='utf-8') as _logf:
        _proc = subprocess.Popen(
            [sys.executable, '-u', TRAIN_SCRIPT],
            cwd=_PROJECT, env=_env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        )
        for _line in _proc.stdout:
            print(_line, end='')
            _logf.write(_line)
            _logf.flush()
            try:
                _babysit(_line)
            except Exception:
                pass  # babysitting is a bonus — never crash training because of it
        _rc = _proc.wait()

    print('\nTraining finished with exit code:', _rc)
    if _rc != 0:
        raise RuntimeError('❌ Training crashed — scroll up to the last red error line and show it to the assistant.')
    print('✅ Training complete — run the next cell to collect your results.')


In [ ]:
# Cell 5 — RESULTS (no edits needed): collects your files and tells you where they go
import os, shutil

if 'RUN' not in globals():
    raise RuntimeError('❌ You skipped the ⭐ cell. Go back and run it first.')

_PROJECT = globals().get('PROJECT', '/kaggle/working/IsotopePINN')
_WORK = globals().get('WORK', '/kaggle/working')

# Where each file should live in your local repo (the "New folder\New folder" project)
_LOCAL_MAP = {
    'A':      {'weights': 'v3_pilstm\\weights\\pi_lstm_best.pth',
               'summary': 'v3_pilstm\\results\\train_summary.json'},
    'B':      {'weights': 'v3_pilstm\\weights\\pi_lstm_best_B.pth',
               'summary': 'v3_pilstm\\results\\train_summary_B.json'},
    'C':      {'weights': 'v3_pilstm\\weights\\baseline_lstm_best.pth',
               'summary': 'v3_pilstm\\results\\baseline_lstm_summary.json'},
    'SMOKE':  {'weights': None, 'summary': None},   # rehearsal — nothing to keep
}
_NEXT = {'SMOKE': 'A', 'A': 'B', 'B': 'C', 'C': 'EVIDENCE', 'EVIDENCE': None}

_downloads = []

if RUN == 'EVIDENCE':
    _ev = ['v3_pilstm/results/compare_v2_pilstm.json',
           'v3_pilstm/results/conformal_validation_cvplus.json',
           'v3_pilstm/results/speed_benchmark.json']
    for _rel in _ev:
        _src = os.path.join(_PROJECT, _rel)
        if os.path.isfile(_src):
            _dst = os.path.join(_WORK, os.path.basename(_rel))
            shutil.copy2(_src, _dst)
            _downloads.append((_dst, 'v3_pilstm\\results\\' + os.path.basename(_rel)))
    if not _downloads:
        print('❌ No evidence files found — run the evidence cell below first.')
    else:
        print('📥 Download these from the Output panel (right side):')
        for _dst, _loc in _downloads:
            print(f'  • {os.path.basename(_dst)}  →  put in your repo at: {_loc}')
        print('\n🎉 All done — you have the full evidence pack for your poster!')
elif RUN == 'SMOKE':
    print('🔬 SMOKE run finished — nothing to download (it was just a rehearsal).')
    print('✅ Your setup works! Now change RUN to "A" in the ⭐ cell and press Run All again.')
else:
    _src_w = os.path.join(_PROJECT, RUN_WEIGHTS)
    _src_s = os.path.join(_PROJECT, RUN_SUMMARY)
    if not os.path.isfile(_src_w):
        raise FileNotFoundError('❌ No weight file found — did the training cell finish successfully?')
    _dst_w = os.path.join(_WORK, f'pi_lstm_best_RUN_{RUN}.pth' if RUN != 'C' else 'baseline_lstm_best_RUN_C.pth')
    shutil.copy2(_src_w, _dst_w)
    _downloads.append((_dst_w, _LOCAL_MAP[RUN]['weights']))
    if os.path.isfile(_src_s):
        _dst_s = os.path.join(_WORK, os.path.basename(RUN_SUMMARY))
        shutil.copy2(_src_s, _dst_s)
        _downloads.append((_dst_s, _LOCAL_MAP[RUN]['summary']))
    _log = os.path.join(_WORK, f'train_log_RUN_{RUN}.txt')
    if os.path.isfile(_log):
        _downloads.append((_log, 'v3_pilstm\\results\\' + os.path.basename(_log)))

    print('✅ RUN', RUN, 'results collected!')
    print('\n📥 Download these from the Output panel (right side of the screen):')
    for _dst, _loc in _downloads:
        print(f'  • {os.path.basename(_dst)}')
        print(f'      → put it in your repo at: {_loc}')

    _nxt = _NEXT.get(RUN)
    if _nxt:
        print(f'\n👉 Next recommended run: change RUN to "{_nxt}" in the ⭐ cell and press Run All.')
    if RUN == 'A':
        print('   (Run B is the control — it proves your new settings are what helped.)')
    elif RUN == 'B':
        print('   (Run C is the no-physics baseline — the fair comparison for your poster.)')
    elif RUN == 'C':
        print('   (EVIDENCE needs this same Kaggle session, or copy the A and C weight files back into a new session.)')


In [ ]:
# Cell 6 — EVIDENCE (no edits needed): builds the poster numbers AFTER runs A, B and C
import json, os, subprocess, sys

if 'RUN' not in globals():
    raise RuntimeError('❌ You skipped the ⭐ cell. Go back and run it first.')

_PROJECT = globals().get('PROJECT', '/kaggle/working/IsotopePINN')

if RUN != 'EVIDENCE':
    print(f'RUN = "{RUN}" — this cell is only for RUN = "EVIDENCE". Nothing to do here yet. ✅')
else:
    _need = {'v3_pilstm/weights/pi_lstm_best.pth': 'Run A',
             'v3_pilstm/weights/baseline_lstm_best.pth': 'Run C',
             'weights/pinn_best_weights.pth': 'the zip'}
    _miss = [f'{lbl} file ({f})' for f, lbl in _need.items()
             if not os.path.isfile(os.path.join(_PROJECT, f))]
    if _miss:
        raise FileNotFoundError('❌ Missing before evidence can run:\n  - ' + '\n  - '.join(_miss))

    _env = os.environ.copy()
    _env.update(RUN_ENV)
    _env['PYTHONUNBUFFERED'] = '1'
    _scripts = [
        ('Compare models (new PI-LSTM vs baseline vs older model)', 'v3_pilstm/analysis/compare_models.py'),
        ('Uncertainty check (conformal cv+, 100/100 scenarios)', 'v3_pilstm/analysis/run_conformal_validation.py'),
        ('Speed benchmark (how fast is the model?)', 'v3_pilstm/scripts/speed_benchmark.py'),
    ]
    for _title, _s in _scripts:
        print('\n' + '=' * 70)
        print('▶', _title)
        print('=' * 70)
        _r = subprocess.run([sys.executable, '-u', _s], cwd=_PROJECT, env=_env)
        if _r.returncode != 0:
            print(f'⚠️ {_s} exited with code {_r.returncode} — show the output above to the assistant.')

    # ---------- poster-ready numbers ----------
    def _load(rel):
        p = os.path.join(_PROJECT, rel)
        return json.load(open(p, encoding='utf-8')) if os.path.isfile(p) else None

    _cmp = _load('v3_pilstm/results/compare_v2_pilstm.json')
    _conf = _load('v3_pilstm/results/conformal_validation_cvplus.json')
    _spd = _load('v3_pilstm/results/speed_benchmark.json')

    print('\n' + '📊' * 20)
    print('POSTER-READY NUMBERS (copy these into your project):')
    print('📊' * 20)

    def _walk(d, out):
        '''Find (median-ish) numbers in nested result dicts, keyed by model name.'''
        if isinstance(d, dict):
            for k, v in d.items():
                if isinstance(v, (int, float)) and not isinstance(v, bool):
                    out.append((k, v))
                else:
                    _walk(v, out)
        elif isinstance(d, list):
            for v in d:
                _walk(v, out)

    if _cmp:
        print('\n— Accuracy (median relative error on Ac-225, lower is better) —')
        _nums = []
        _walk(_cmp, _nums)
        _seen = set()
        for k, v in _nums:
            kl = k.lower()
            if ('med' in kl or 'median' in kl) and kl not in _seen:
                _seen.add(kl)
                print(f'  {k}: {v:.4g}')
    if _conf:
        print('\n— Uncertainty (conformal prediction intervals) —')
        _nums = []
        _walk(_conf, _nums)
        _seen = set()
        for k, v in _nums:
            kl = k.lower()
            if any(w in kl for w in ('coverage', 'width', 'alpha')) and kl not in _seen:
                _seen.add(kl)
                print(f'  {k}: {v:.4g}')
    if _spd:
        print('\n— Speed —')
        _nums = []
        _walk(_spd, _nums)
        _seen = set()
        for k, v in _nums:
            kl = k.lower()
            if any(w in kl for w in ('latency', 'throughput', 'median_ms', 'per_s')) and kl not in _seen:
                _seen.add(kl)
                print(f'  {k}: {v:.4g}')

    if not any((_cmp, _conf, _spd)):
        print('⚠️ Could not read the result JSONs — the raw numbers are in the output above.')

    print('\n👉 Now run the RESULTS cell above (Cell 5) to collect the evidence files for download.')
